# Intrusion Detection System — Feature Engineering
### Dataset: CICIDS-2017  ·  Runs on **Kaggle**

Runs *after* `Preprocessing.ipynb`. Takes the cleaned `train.parquet` / `test.parquet`
and produces **model-ready feature matrices**.

**Kaggle setup:**
1. Create a Kaggle **Dataset** named `ids-preprocessed` containing `train.parquet`,
   `test.parquet`, `label_mapping.csv` (the output of `Preprocessing.ipynb`).
2. New Kaggle Notebook → *Add Input* → attach that dataset.
3. **Settings → Internet: ON** (needed for `pip install imbalanced-learn`).
4. Outputs are written to `/kaggle/working/` (persistent, downloadable as a zip).

> No raw dataset needed — `Init_Win_bytes` sentinel handling + `has_init_win_*` flags
> are already done in `Preprocessing.ipynb`. This notebook only consumes the parquet files.

**Pipeline:**
| # | Step | Notes |
|---|------|-------|
| 1 | Skew correction | `log1p` transform on highly-skewed features (\|skew\|>1) |
| 2 | Derived features | meaningful ratios / rates from existing columns |
| 3 | Scaling | `StandardScaler` — **fit on train only**, apply to test |
| 4 | Feature ranking | mutual information + RF importance, for reference |
| 5 | Class balancing | `SMOTE` — **train set only**, test keeps real-world distribution |
| 6 | Save | engineered train/test matrices + fitted transformers |

**Golden rule:** every transformer is *fit on training data only* and then applied to the test set — no leakage.

**Outputs (to `/kaggle/working/`):**
- `train_engineered.parquet` / `test_engineered.parquet` — scaled features (no SMOTE)
- `train_binary_smote.parquet` / `train_multi_smote.parquet` — SMOTE-balanced training sets
- `scaler.joblib`, `feature_list.json`, `feature_ranking.csv`
- `FeatureEngineering_Report.txt` + `figures/`

## 1. Imports, Config & Report Helpers

In [ ]:
!pip install -q imbalanced-learn

import os, json, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.bbox'] = 'tight'

RANDOM_SEED    = 42
SKEW_THRESHOLD = 1.0          # |skew| above this gets log1p-transformed

# ── Kaggle paths ───────────────────────────────────────────────────
# INPUT  : the Kaggle Dataset holding train.parquet / test.parquet / label_mapping.csv
#          (uploaded from Preprocessing.ipynb output). Name the dataset 'ids-preprocessed'.
# OUTPUT : /kaggle/working is persistent and downloadable as a zip after the run.
PRE_DIR     = '/kaggle/input/ids-preprocessed'
OUT_DIR     = '/kaggle/working'
FIGURES_DIR = os.path.join(OUT_DIR, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── report helpers ─────────────────────────────────────────────────
_report_lines = []

def _log(text=''):
    _report_lines.append(str(text))
    print(text)

def _savefig(name, fig=None):
    path = os.path.join(FIGURES_DIR, name)
    (fig or plt).savefig(path, dpi=130, bbox_inches='tight')
    return path

def write_report():
    path = os.path.join(OUT_DIR, 'FeatureEngineering_Report.txt')
    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(_report_lines))
    print(f'\nReport saved -> {path}')

_log('=' * 70)
_log('FEATURE ENGINEERING REPORT  —  CICIDS-2017')
_log(f'Generated : {datetime.now().strftime("%Y-%m-%d %H:%M")}')
_log('=' * 70)
print('\nSetup complete.')
print('  Reading from :', PRE_DIR)
print('  Writing to   :', OUT_DIR)

## 2. Load Cleaned Train/Test Splits
Produced by `Preprocessing.ipynb`. If the parquet files are not in `/content/preprocessed`, mount Drive / re-run preprocessing first.

In [ ]:
train_path = os.path.join(PRE_DIR, 'train.parquet')
test_path  = os.path.join(PRE_DIR, 'test.parquet')

if not (os.path.exists(train_path) and os.path.exists(test_path)):
    raise FileNotFoundError(
        f'Cleaned splits not found in {PRE_DIR}. Run Preprocessing.ipynb first '
        '(or copy train.parquet / test.parquet there).')

train_df = pd.read_parquet(train_path)
test_df  = pd.read_parquet(test_path)

# columns that are NOT features
META_COLS   = ['Label', 'Label_grouped', 'source_file', 'label_binary', 'label_multi']
feature_cols = [c for c in train_df.columns if c not in META_COLS]

_log('')
_log('── STEP 0 : LOADED CLEANED SPLITS ─────────────────────────')
_log(f'  Train : {train_df.shape[0]:,} rows x {train_df.shape[1]} cols')
_log(f'  Test  : {test_df.shape[0]:,} rows x {test_df.shape[1]} cols')
_log(f'  Feature columns : {len(feature_cols)}')
print('\nFeature columns:')
print(feature_cols)

## 3. Skew Correction (log1p transform)
EDA found **68 features with |skew| > 1**. Heavy right-skew hurts distance-based and linear models. We apply `log1p` to features whose **training-set** skew exceeds the threshold.

- Threshold and the list of transformed columns are decided on **train only**, then the *same* columns are transformed in test.
- Flags / count columns and the already-bounded `has_init_win_*` are excluded.
- `log1p(x) = log(1+x)` needs non-negative input — preprocessing already clipped negatives, except the `Init_Win` sentinel (`-1`), which we shift by `+1` before transforming.

In [ ]:
# columns we never log-transform (binary flags / small-range counts)
NO_TRANSFORM = {
    'has_init_win_fwd', 'has_init_win_bwd',
    'FIN Flag Count', 'SYN Flag Count', 'PSH Flag Count', 'ACK Flag Count',
    'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio',
    'Destination Port',
}
# Init_Win cols can be -1; shift by +1 so log1p input stays >= 0
SHIFT_COLS = {'Init_Win_bytes_forward', 'Init_Win_bytes_backward'}

candidate_cols = [c for c in feature_cols if c not in NO_TRANSFORM]

# decide which to transform — based on TRAIN skew only
train_skew = train_df[candidate_cols].skew()
skewed_cols = train_skew[train_skew.abs() > SKEW_THRESHOLD].index.tolist()

_log('')
_log('── STEP 2 : SKEW CORRECTION (log1p) ───────────────────────')
_log(f'  Skew threshold        : |skew| > {SKEW_THRESHOLD}')
_log(f'  Candidate features    : {len(candidate_cols)}')
_log(f'  Features transformed  : {len(skewed_cols)}')

skew_before = train_df[skewed_cols].skew()

def apply_log1p(df_split):
    df_split = df_split.copy()
    for c in skewed_cols:
        x = df_split[c].astype(float)
        if c in SHIFT_COLS:
            x = x + 1                      # -1 sentinel -> 0
        x = x.clip(lower=0)                # safety against any residual negatives
        df_split[c] = np.log1p(x)
    return df_split

train_df = apply_log1p(train_df)
test_df  = apply_log1p(test_df)

skew_after = train_df[skewed_cols].skew()
skew_cmp = pd.DataFrame({'skew_before': skew_before, 'skew_after': skew_after})
skew_cmp['abs_improvement'] = skew_cmp['skew_before'].abs() - skew_cmp['skew_after'].abs()
skew_cmp = skew_cmp.sort_values('abs_improvement', ascending=False)

_log(f'  Mean |skew| before    : {skew_before.abs().mean():.2f}')
_log(f'  Mean |skew| after     : {skew_after.abs().mean():.2f}')
_log('  Per-feature skew change:')
_log(skew_cmp.to_string())
display(skew_cmp)

# before/after plot
fig, axes = plt.subplots(1, 2, figsize=(18, max(5, len(skewed_cols) * 0.25)))
y = np.arange(len(skew_cmp))
axes[0].barh(y, skew_cmp['skew_before'], color='#F44336')
axes[0].barh(y, skew_cmp['skew_after'],  color='#4CAF50', alpha=0.8)
axes[0].set_yticks(y); axes[0].set_yticklabels(skew_cmp.index, fontsize=6)
axes[0].axvline(1,  color='gray', ls='--', lw=0.8)
axes[0].axvline(-1, color='gray', ls='--', lw=0.8)
axes[0].set_title('Skewness — Before (red) vs After (green)')
axes[0].set_xlabel('Skewness')

axes[1].bar(['Before', 'After'],
            [skew_before.abs().mean(), skew_after.abs().mean()],
            color=['#F44336', '#4CAF50'])
axes[1].set_title('Mean |Skewness| across transformed features')
axes[1].set_ylabel('Mean |skew|')
for i, v in enumerate([skew_before.abs().mean(), skew_after.abs().mean()]):
    axes[1].text(i, v, f'{v:.2f}', ha='center', va='bottom')
plt.tight_layout()
_savefig('02_skew_before_after.png', fig)
plt.show()

In [ ]:
# distribution shape for the 6 features with the biggest skew improvement
show = skew_cmp.head(6).index.tolist()
fig, axes = plt.subplots(2, 6, figsize=(24, 7))
for j, col in enumerate(show):
    # 'before' = invert the log1p just for display
    raw_vals = np.expm1(train_df[col].sample(min(20000, len(train_df)), random_state=1))
    log_vals = train_df[col].sample(min(20000, len(train_df)), random_state=1)
    axes[0, j].hist(raw_vals.clip(upper=raw_vals.quantile(0.99)), bins=50,
                    color='#F44336', alpha=0.8)
    axes[0, j].set_title(f'{col}\n(original)', fontsize=7)
    axes[1, j].hist(log_vals, bins=50, color='#4CAF50', alpha=0.8)
    axes[1, j].set_title(f'{col}\n(log1p)', fontsize=7)
    for r in (0, 1):
        axes[r, j].tick_params(labelsize=6)
axes[0, 0].set_ylabel('original', fontsize=9)
axes[1, 0].set_ylabel('log1p', fontsize=9)
plt.suptitle('Distribution shape — original vs log1p (top-6 most improved)', fontsize=12)
plt.tight_layout()
_savefig('03_skew_distributions.png', fig)
plt.show()

## 5. Derived Features
Create a small set of **interpretable ratio / rate features** from existing columns. These often help tree models split cleanly (e.g. asymmetry between forward and backward traffic is a strong attack signal).

All derived features are computed identically on train and test (no fitting involved), with safe division (`/0 -> 0`).

In [ ]:
def safe_div(a, b):
    return np.where(np.abs(b) < 1e-9, 0.0, a / b)

def add_derived(df_split):
    df_split = df_split.copy()
    # forward / backward byte asymmetry
    df_split['fwd_bwd_byte_ratio'] = safe_div(df_split['Subflow Fwd Bytes'],
                                              df_split['Subflow Bwd Bytes'])
    # forward / backward packet-rate asymmetry
    df_split['fwd_bwd_pkts_ratio'] = safe_div(df_split['Fwd Packets/s'],
                                              df_split['Bwd Packets/s'])
    # average segment-size asymmetry
    df_split['seg_size_ratio']     = safe_div(df_split['Avg Fwd Segment Size'],
                                              df_split['Avg Bwd Segment Size'])
    # IAT (inter-arrival time) asymmetry
    df_split['fwd_bwd_iat_ratio']  = safe_div(df_split['Fwd IAT Mean'],
                                              df_split['Bwd IAT Mean'])
    # header overhead: how much of the traffic is header bytes
    df_split['bwd_header_ratio']   = safe_div(df_split['Bwd Header Length'],
                                              df_split['Subflow Bwd Bytes'])
    # active / idle behaviour ratio
    df_split['active_idle_ratio']  = safe_div(df_split['Active Mean'],
                                              df_split['Idle Std'] + df_split['Idle Min'])
    # total flag activity (sum of all TCP flag counts)
    flag_cols = ['FIN Flag Count', 'SYN Flag Count', 'PSH Flag Count', 'ACK Flag Count',
                 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count']
    df_split['total_flag_count']   = df_split[flag_cols].sum(axis=1)
    return df_split

DERIVED_COLS = ['fwd_bwd_byte_ratio', 'fwd_bwd_pkts_ratio', 'seg_size_ratio',
                'fwd_bwd_iat_ratio', 'bwd_header_ratio', 'active_idle_ratio',
                'total_flag_count']

train_df = add_derived(train_df)
test_df  = add_derived(test_df)

# any inf produced by ratios -> 0 (safe_div already guards, this is belt-and-braces)
for c in DERIVED_COLS:
    train_df[c] = train_df[c].replace([np.inf, -np.inf], 0).fillna(0)
    test_df[c]  = test_df[c].replace([np.inf, -np.inf], 0).fillna(0)

feature_cols = feature_cols + DERIVED_COLS

_log('')
_log('── STEP 3 : DERIVED FEATURES ──────────────────────────────')
_log(f'  Added {len(DERIVED_COLS)} derived features:')
for c in DERIVED_COLS:
    _log(f'    - {c}')
_log(f'  Total feature count now : {len(feature_cols)}')

# plot: how well each derived feature separates BENIGN vs ATTACK (median per class)
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()
for i, c in enumerate(DERIVED_COLS):
    grp = train_df.groupby('label_binary')[c].median()
    axes[i].bar(['BENIGN', 'ATTACK'], [grp.get(0, 0), grp.get(1, 0)],
                color=['#2196F3', '#F44336'])
    axes[i].set_title(c, fontsize=8)
    axes[i].tick_params(labelsize=7)
axes[-1].set_axis_off()
plt.suptitle('Derived features — median value, BENIGN vs ATTACK', fontsize=12)
plt.tight_layout()
_savefig('04_derived_features.png', fig)
plt.show()

## 6. Feature Scaling — StandardScaler
`StandardScaler` standardises every feature to **zero mean / unit variance**. Required for logistic regression, SVM and the from-scratch neural net.

**Leakage guard:** the scaler is `fit` on the **training matrix only**; the *same* fitted scaler `transform`s the test matrix.

In [ ]:
# assemble feature matrices (X) and the two label vectors (y)
X_train = train_df[feature_cols].astype(float).copy()
X_test  = test_df[feature_cols].astype(float).copy()

y_train_bin   = train_df['label_binary'].values
y_test_bin    = test_df['label_binary'].values
y_train_multi = train_df['label_multi'].values
y_test_multi  = test_df['label_multi'].values

# final safety: no NaN / inf should remain
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test  = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train),
                              columns=feature_cols, index=X_train.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),
                              columns=feature_cols, index=X_test.index)

_log('')
_log('── STEP 4 : SCALING (StandardScaler) ──────────────────────')
_log(f'  Fit on TRAIN only ({X_train.shape[0]:,} rows), applied to test.')
_log(f'  Train mean (post-scale) ~ {X_train_scaled.mean().mean():+.4f}  '
     f'std ~ {X_train_scaled.std().mean():.4f}')
_log(f'  Test  mean (post-scale) ~ {X_test_scaled.mean().mean():+.4f}  '
     f'std ~ {X_test_scaled.std().mean():.4f}   (test != exactly 0/1 — expected, no leakage)')

# plot: feature means/std before vs after scaling (sample of features)
sample_feats = feature_cols[:25]
fig, axes = plt.subplots(1, 2, figsize=(20, 5))
axes[0].bar(np.arange(len(sample_feats)) - 0.2, X_train[sample_feats].mean(), 0.4,
            label='before', color='#FF9800')
axes[0].bar(np.arange(len(sample_feats)) + 0.2, X_train_scaled[sample_feats].mean(), 0.4,
            label='after', color='#4CAF50')
axes[0].set_title('Feature MEAN — before vs after scaling (first 25 feats)')
axes[0].set_xticks(range(len(sample_feats)))
axes[0].set_xticklabels(sample_feats, rotation=90, fontsize=6)
axes[0].legend()
axes[0].set_yscale('symlog')

axes[1].bar(np.arange(len(sample_feats)) - 0.2, X_train[sample_feats].std(), 0.4,
            label='before', color='#FF9800')
axes[1].bar(np.arange(len(sample_feats)) + 0.2, X_train_scaled[sample_feats].std(), 0.4,
            label='after', color='#4CAF50')
axes[1].set_title('Feature STD — before vs after scaling (first 25 feats)')
axes[1].set_xticks(range(len(sample_feats)))
axes[1].set_xticklabels(sample_feats, rotation=90, fontsize=6)
axes[1].legend()
axes[1].set_yscale('symlog')
plt.tight_layout()
_savefig('05_scaling_before_after.png', fig)
plt.show()

## 7. Feature Ranking (Mutual Information + RF Importance)
*Reference only* — we are **not** dropping features here, just understanding which ones carry signal. Computed on a stratified sample of the scaled training set for speed.

Two views:
- **Mutual information** vs the binary label — model-agnostic dependency
- **Random-Forest importance** vs the multi-class label — captures interactions

In [ ]:
# stratified sample for speed (MI + RF on 2M rows is slow)
SAMPLE_N = 150_000
samp = (X_train_scaled
        .assign(_y=y_train_multi)
        .groupby('_y', group_keys=False)
        .apply(lambda g: g.sample(min(len(g), max(1, SAMPLE_N * len(g) // len(X_train_scaled))),
                                  random_state=RANDOM_SEED)))
samp_y_multi = samp['_y'].values
samp_X = samp.drop(columns='_y')
samp_y_bin = (samp_y_multi != 0).astype(int)

# mutual information vs binary label
mi = mutual_info_classif(samp_X, samp_y_bin, random_state=RANDOM_SEED)
mi_s = pd.Series(mi, index=feature_cols).sort_values(ascending=False)

# random forest importance vs multi-class label
rf = RandomForestClassifier(n_estimators=120, max_depth=18, n_jobs=-1,
                            class_weight='balanced', random_state=RANDOM_SEED)
rf.fit(samp_X, samp_y_multi)
rf_s = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

ranking = pd.DataFrame({'mutual_info': mi_s, 'rf_importance': rf_s}).fillna(0)
ranking['rank_mi'] = ranking['mutual_info'].rank(ascending=False)
ranking['rank_rf'] = ranking['rf_importance'].rank(ascending=False)
ranking = ranking.sort_values('rf_importance', ascending=False)

_log('')
_log('── STEP 5 : FEATURE RANKING (reference) ───────────────────')
_log(f'  Computed on stratified sample of {len(samp_X):,} rows')
_log('  Top 15 features by RF importance:')
_log(ranking.head(15).to_string())
display(ranking.head(20))

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
mi_s.head(20).iloc[::-1].plot(kind='barh', ax=axes[0], color='teal')
axes[0].set_title('Top 20 — Mutual Information (vs binary label)')
axes[0].set_xlabel('MI score')
rf_s.head(20).iloc[::-1].plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('Top 20 — Random Forest Importance (vs multi-class)')
axes[1].set_xlabel('Importance')
plt.suptitle('Feature Ranking — reference only, no features dropped', fontsize=13)
plt.tight_layout()
_savefig('06_feature_ranking.png', fig)
plt.show()

## 8. Class Balancing — SMOTE (training set only)
The training set is highly imbalanced (BENIGN ≈ 83%). **SMOTE** synthesises minority-class samples so the model is not biased toward BENIGN.

**Critical rules:**
- SMOTE is applied to the **training set only**. The test set keeps the real-world distribution so metrics reflect reality.
- SMOTE runs on the **scaled** features (it interpolates — features must be on comparable scales).
- We build **two** balanced training sets: one for the binary task, one for the multi-class task.

For the multi-class case we don't force perfectly equal classes (that would synthesise 2M Web-Attack rows from ~1.7k real ones); instead we lift each minority class up to a capped target.

In [ ]:
_log('')
_log('── STEP 6 : SMOTE BALANCING (train only) ──────────────────')

# ---- binary ----
sm_bin = SMOTE(random_state=RANDOM_SEED, k_neighbors=5)
Xb, yb = sm_bin.fit_resample(X_train_scaled, y_train_bin)
_log('  BINARY:')
_log(f'    before : {dict(pd.Series(y_train_bin).value_counts().sort_index())}')
_log(f'    after  : {dict(pd.Series(yb).value_counts().sort_index())}')

# ---- multi-class ----
# cap each minority class at min(majority, 200k) synthetic target
counts = pd.Series(y_train_multi).value_counts()
majority_n = counts.max()
TARGET = min(majority_n, 200_000)
sampling_strategy = {cls: max(n, TARGET) for cls, n in counts.items() if cls != counts.idxmax()}
# keep majority untouched
sm_multi = SMOTE(random_state=RANDOM_SEED, k_neighbors=5,
                 sampling_strategy=sampling_strategy)
Xm, ym = sm_multi.fit_resample(X_train_scaled, y_train_multi)
_log('  MULTI-CLASS:')
_log(f'    target per minority class : {TARGET:,}')
_log(f'    before : {dict(counts.sort_index())}')
_log(f'    after  : {dict(pd.Series(ym).value_counts().sort_index())}')

# plot before/after for both tasks
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

b_df = pd.DataFrame({'before': pd.Series(y_train_bin).value_counts().sort_index(),
                     'after':  pd.Series(yb).value_counts().sort_index()})
b_df.index = ['BENIGN', 'ATTACK']
b_df.plot(kind='bar', ax=axes[0], color=['#FF9800', '#4CAF50'])
axes[0].set_title('Binary — class counts before vs after SMOTE')
axes[0].tick_params(axis='x', rotation=0)

m_df = pd.DataFrame({'before': counts.sort_index(),
                     'after':  pd.Series(ym).value_counts().sort_index()})
m_df.plot(kind='bar', ax=axes[1], color=['#FF9800', '#4CAF50'], logy=True)
axes[1].set_title('Multi-class — class counts before vs after SMOTE (log scale)')
axes[1].set_xlabel('label_multi')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('SMOTE applied to TRAINING set only', fontsize=13)
plt.tight_layout()
_savefig('07_smote_before_after.png', fig)
plt.show()

## 9. Save Engineered Datasets & Transformers

In [ ]:
# 1) scaled (non-SMOTE) train/test — keep both labels for flexibility downstream
train_eng = X_train_scaled.copy()
train_eng['label_binary'] = y_train_bin
train_eng['label_multi']  = y_train_multi
test_eng = X_test_scaled.copy()
test_eng['label_binary']  = y_test_bin
test_eng['label_multi']   = y_test_multi

p1 = os.path.join(OUT_DIR, 'train_engineered.parquet')
p2 = os.path.join(OUT_DIR, 'test_engineered.parquet')
train_eng.to_parquet(p1, index=False)
test_eng.to_parquet(p2, index=False)

# 2) SMOTE-balanced training sets (one per task)
bin_smote = Xb.copy(); bin_smote['label_binary'] = yb
multi_smote = Xm.copy(); multi_smote['label_multi'] = ym
p3 = os.path.join(OUT_DIR, 'train_binary_smote.parquet')
p4 = os.path.join(OUT_DIR, 'train_multi_smote.parquet')
bin_smote.to_parquet(p3, index=False)
multi_smote.to_parquet(p4, index=False)

# 3) fitted transformer + metadata
p5 = os.path.join(OUT_DIR, 'scaler.joblib')
joblib.dump(scaler, p5)
p6 = os.path.join(OUT_DIR, 'feature_list.json')
with open(p6, 'w') as f:
    json.dump({'feature_cols': feature_cols,
               'log1p_cols': skewed_cols,
               'derived_cols': DERIVED_COLS,
               'shift_cols': sorted(SHIFT_COLS)}, f, indent=2)
p7 = os.path.join(OUT_DIR, 'feature_ranking.csv')
ranking.to_csv(p7)

_log('')
_log('── STEP 7 : FILES SAVED ───────────────────────────────────')
for p in [p1, p2, p3, p4, p5, p6, p7]:
    size = os.path.getsize(p) / 1e6
    _log(f'  {os.path.basename(p):32} {size:8.1f} MB')
    print(f'  {os.path.basename(p):32} {size:8.1f} MB')

## 10. Final Summary & Report

In [ ]:
_log('')
_log('=' * 70)
_log('SUMMARY  —  FEATURE ENGINEERING')
_log('=' * 70)
_log(f'  Input feature count   : {len(feature_cols) - len(DERIVED_COLS) - 2}  (from preprocessing)')
_log(f'  + Init_Win flags      : 2')
_log(f'  + Derived features    : {len(DERIVED_COLS)}')
_log(f'  = Final feature count : {len(feature_cols)}')
_log('')
_log(f'  log1p-transformed     : {len(skewed_cols)} features')
_log(f'  Scaling               : StandardScaler (fit on train only)')
_log(f'  Train rows (scaled)   : {len(train_eng):,}')
_log(f'  Test  rows (scaled)   : {len(test_eng):,}')
_log(f'  Train rows (binary SMOTE)      : {len(bin_smote):,}')
_log(f'  Train rows (multi-class SMOTE) : {len(multi_smote):,}')
_log('')
_log('  Steps applied:')
_log('    1. Restored Init_Win -1 sentinel + added has_init_win flags')
_log('    2. log1p transform on highly-skewed features')
_log('    3. Added 7 derived ratio/rate features')
_log('    4. StandardScaler (fit train, apply test)')
_log('    5. Feature ranking (MI + RF) — reference only')
_log('    6. SMOTE on training set only (binary + multi-class)')
_log('    7. Saved engineered datasets + scaler + metadata')
_log('')
_log('  Output files:')
_log('    train_engineered.parquet / test_engineered.parquet  (scaled, real distribution)')
_log('    train_binary_smote.parquet / train_multi_smote.parquet  (balanced training)')
_log('    scaler.joblib, feature_list.json, feature_ranking.csv')
_log('')
_log('  Next step -> Modelling: binary first (use *_binary_smote for training,')
_log('               test_engineered for evaluation), then multi-class.')

FIGURE_INDEX = [
    ('01_init_win_restore.png',    'Init_Win sentinel pattern per attack family'),
    ('02_skew_before_after.png',   'Skewness before/after log1p'),
    ('03_skew_distributions.png',  'Distribution shape original vs log1p'),
    ('04_derived_features.png',    'Derived features — BENIGN vs ATTACK medians'),
    ('05_scaling_before_after.png','Feature mean/std before vs after scaling'),
    ('06_feature_ranking.png',     'Mutual information + RF importance'),
    ('07_smote_before_after.png',  'Class counts before/after SMOTE'),
]
_log('')
_log('  Figures:')
for fname, desc in FIGURE_INDEX:
    _log(f'    {fname:<32} {desc}')

write_report()

print('\n' + '=' * 55)
print('FEATURE ENGINEERING COMPLETE')
print('=' * 55)
print(f'  Final feature count : {len(feature_cols)}')
print(f'  Train / Test (real) : {len(train_eng):,} / {len(test_eng):,}')
print(f'  Binary SMOTE train  : {len(bin_smote):,}')
print(f'  Multi  SMOTE train  : {len(multi_smote):,}')
print(f'  Report  -> {OUT_DIR}/FeatureEngineering_Report.txt')
print(f'  Figures -> {FIGURES_DIR}/  ({len(FIGURE_INDEX)} figures)')